In [1]:
import pandas as pd
import requests 
from bs4 import BeautifulSoup

In [2]:
headers={'User-Agent':'Mozilla/5.0 (Windows NT 6.3; Win 64 ; x64) Apple WeKit /537.36(KHTML , like Gecko) Chrome/80.0.3987.162 Safari/537.36'}
webpage = requests.get('https://www.ambitionbox.com/list-of-companies?page=1', headers=headers)

In [3]:
soup = BeautifulSoup(webpage.text, 'lxml')

## Extracting the individual box of a company. So that we can have entire info of company in one box (name, ratings, jobs, etc)

In [4]:
company = soup.find_all('div', class_='companyCardWrapper')

In [5]:
len(company)

20

In [6]:
company[0]

<div class="companyCardWrapper" itemprop="itemListElement" itemscope="itemscope" itemtype="http://schema.org/ListItem"><meta content="1" itemprop="position"/> <meta content="TCS" itemprop="name"/> <meta content="Tata Consultancy Services" itemprop="alternateName"/> <meta content="https://www.ambitionbox.com/overview/tcs-overview" itemprop="url"/> <meta content="https://static.ambitionbox.com/alpha/company/photos/logos/tcs.jpg" itemprop="image"/> <div class="companyCardWrapper__primaryInformation"><div class="companyCardWrapper__companyLogo"><img alt="Tata Consultancy Services logo" height="50" loading="lazy" onerror="this.onerror=null;this.src='/static/icons/company-placeholder.svg';" src="https://static.ambitionbox.com/assets/v2/images/rs:fit:200:200:false:false/aHR0cHM6Ly9tZWRpYS5uYXVrcmkuY29tL21lZGlhL2FiY29tcGxvZ28vdGNzLW9yaWdpbmFsLmpwZw.webp" width="50"/></div> <div class="companyCardWrapper__metaInformation"><div class="companyCardWrapper__header"><div class="companyCardWrapper__c

## Extracting names

In [10]:
company[0].find_all('h2')

[<h2 class="companyCardWrapper__companyName" title="TCS">
 									TCS
 								</h2>]

In [15]:
names = []
for i in company:
    names.append(i.find('h2').text.strip())

In [16]:
names

['TCS',
 'Accenture',
 'Wipro',
 'Cognizant',
 'Capgemini',
 'HDFC Bank',
 'Infosys',
 'HCLTech',
 'ICICI Bank',
 'Tech Mahindra',
 'Genpact',
 'TP',
 'Jio',
 'Axis Bank',
 'Concentrix Corporation',
 'Amazon',
 'Reliance Retail',
 'iEnergizer',
 'LTIMindtree',
 'IBM']

## Extracting ratings

In [21]:
company[0].find('div', class_='rating_text rating_text--md').text

'\n\t\t\t3.3'

In [24]:
ratings = []
for i in company:
    ratings.append(i.find('div', class_='rating_text rating_text--md').text.strip())

In [25]:
ratings

['3.3',
 '3.7',
 '3.6',
 '3.7',
 '3.6',
 '3.8',
 '3.5',
 '3.4',
 '4.0',
 '3.4',
 '3.6',
 '3.8',
 '4.4',
 '3.6',
 '3.6',
 '3.9',
 '3.9',
 '4.6',
 '3.6',
 '3.9']

## Extracting Type of company

In [32]:
company[0].find('span', class_='companyCardWrapper__interLinking').text.strip().split('|')[0].strip()

'IT Services & Consulting'

In [33]:
ctype = []
for i in company:
    ctype.append(i.find('span', class_='companyCardWrapper__interLinking').text.strip().split('|')[0].strip())

In [36]:
ctype

['IT Services & Consulting',
 'IT Services & Consulting',
 'IT Services & Consulting',
 'IT Services & Consulting',
 'IT Services & Consulting',
 'Banking',
 'IT Services & Consulting',
 'IT Services & Consulting',
 'Banking',
 'IT Services & Consulting',
 'Analytics & KPO',
 'BPO',
 'Telecom',
 'Banking',
 'BPO',
 'Internet',
 'Retail',
 'BPO',
 'IT Services & Consulting',
 'IT Services & Consulting']

## Extracting Highly rated for and critically rated for.

In [39]:
company[0].find_all('span', class_='companyCardWrapper__ratingValues')

[<span class="companyCardWrapper__ratingValues">Job Security</span>,
 <span class="companyCardWrapper__ratingValues">Promotions, Salary, Work Satisfaction</span>]

In [40]:
import numpy as np

In [41]:
highly_rated = []
critically_rated = []
for i in company:
    try:
        highly_rated.append(i.find_all('span', class_='companyCardWrapper__ratingValues')[0].text.strip())
    except:
        highly_rated.append(np.nan)
    try:
        critically_rated.append(i.find_all('span', class_='companyCardWrapper__ratingValues')[1].text.strip())
    except:
        critically_rated.append(np.nan)

In [44]:
len(highly_rated)

20

# Lets make df from these 20 company data

In [45]:
data_20 = {'name':names, 'rating':ratings, 'ctype':ctype, 'highly_rated':highly_rated, 'critically_rated':critically_rated}

In [46]:
df_20 = pd.DataFrame(data_20)

In [47]:
df_20

,name,rating,ctype,highly_rated,critically_rated
0,TCS,3.3,IT Services & Consulting,Job Security,"Promotions, Salary, Work Satisfaction"
1,Accenture,3.7,IT Services & Consulting,"Promotions, Salary, Work Satisfaction",NaN
2,Wipro,3.6,IT Services & Consulting,"Promotions, Salary, Work Satisfaction",NaN
3,Cognizant,3.7,IT Services & Consulting,"Promotions, Salary, Work Satisfaction",NaN
4,Capgemini,3.6,IT Services & Consulting,"Work Life Balance, Job Security","Promotions, Salary, Work Satisfaction"
5,HDFC Bank,3.8,Banking,"Job Security, Skill Development","Promotions, Salary"
6,Infosys,3.5,IT Services & Consulting,Job Security,"Promotions, Salary, Work Satisfaction"
7,HCLTech,3.4,IT Services & Consulting,"Promotions, Salary, Work Satisfaction",NaN
8,ICICI Bank,4.0,Banking,"Job Security, Promotions, Skill Development",NaN
9,Tech Mahindra,3.4,IT Services & Consulting,"Promotions, Salary, Work Satisfaction",NaN


# Lets do for 500 pages. 

In [52]:
final_df = pd.DataFrame()

for j in range(1, 501):
    url = 'https://www.ambitionbox.com/list-of-companies?page={}'.format(j)

    headers={'User-Agent':'Mozilla/5.0 (Windows NT 6.3; Win 64 ; x64) Apple WeKit /537.36(KHTML , like Gecko) Chrome/80.0.3987.162 Safari/537.36'}
    webpage = requests.get(url, headers=headers)

    soup = BeautifulSoup(webpage.text, 'lxml')

    company = soup.find_all('div', class_='companyCardWrapper')
    
    names = []
    ratings = []
    ctype = []
    highly_rated = []
    critically_rated = []
    for i in company:
        # Names
        names.append(i.find('h2').text.strip())
        
        # ratings
        try:
            ratings.append(i.find('div', class_='rating_text rating_text--md').text.strip())
        except:
            ratings.append(np.nan)
        
        # company type
        try:
            ctype.append(i.find('span', class_='companyCardWrapper__interLinking').text.strip().split('|')[0].strip())
        except:
            ctype.append(np.nan)
        
        # Highly rated for
        try:
            highly_rated.append(i.find_all('span', class_='companyCardWrapper__ratingValues')[0].text.strip())
        except:
            highly_rated.append(np.nan)
        
        # Critically rated for
        try:
            critically_rated.append(i.find_all('span', class_='companyCardWrapper__ratingValues')[1].text.strip())
        except:
            critically_rated.append(np.nan)

    data_20 = {'name':names, 'rating':ratings, 'ctype':ctype, 'highly_rated':highly_rated, 'critically_rated':critically_rated}
    df_20 = pd.DataFrame(data_20)

    final_df = pd.concat([final_df, df_20], ignore_index=True)

In [61]:
final_df

,name,rating,ctype,highly_rated,critically_rated
0,TCS,3.3,IT Services & Consulting,Job Security,"Promotions, Salary, Work Satisfaction"
1,Accenture,3.7,IT Services & Consulting,"Promotions, Salary, Work Satisfaction",NaN
2,Wipro,3.6,IT Services & Consulting,"Promotions, Salary, Work Satisfaction",NaN
3,Cognizant,3.7,IT Services & Consulting,"Promotions, Salary, Work Satisfaction",NaN
4,Capgemini,3.6,IT Services & Consulting,"Work Life Balance, Job Security","Promotions, Salary, Work Satisfaction"
...,...,...,...,...,...
9995,BBD University,3.6,Financial Services,"Promotions, Salary, Work Satisfaction",NaN
9996,Dlecta Foods,3.8,Food Processing,Skill Development,"Company Culture, Promotions"
9997,Autocop India,3.6,Semiconductors,"Company Culture, Promotions, Skill Development",NaN
9998,Techwise Digital,4.6,IT Services & Consulting,"Company Culture, Work Life Balance, Work Satis...",NaN


In [62]:
final_df.to_csv('company.csv', index=False)